In [ ]:

%pip -q install -U kagglehub opencv-python-headless scikit-learn seaborn

import tensorflow as tf
print("TensorFlow:", tf.__version__)

In [ ]:

import os
import random
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from IPython.display import display
from tensorflow.keras.applications import InceptionV3, EfficientNetB0, DenseNet121, ResNet50, ResNet101

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("TF version:", tf.__version__)

In [ ]:

IMG_SIZE = (150, 150)
IMG_CHANNELS = 3
INPUT_SHAPE = (*IMG_SIZE, IMG_CHANNELS)
BATCH_SIZE = 32
EPOCHS = 40
LEARNING_RATE = None  # None = Adam() default, because the paper does not state a custom learning rate
DROPOUT = 0.5
CLAHE_CLIP = 3.0
CLAHE_GRID = (8, 8)
GAMMA = 1.3
GAMMA_VALUES_FOR_TUNING = [0.8, 1.0, 1.3]


PREPROCESS_TEST = False

CLASS_NAMES = ["CONTROL", "AD", "PD"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

TARGET_COUNTS = {
    "AD":      {"train": 2560, "validation": 320, "test": 320},
    "CONTROL": {"train": 2937, "validation": 367, "test": 368},
    "PD":      {"train": 774,  "validation": 97,  "test": 96},
}

print(json.dumps(TARGET_COUNTS, indent=2))

In [ ]:

import kagglehub

DATASET_ID = "farjanakabirsamanta/alzheimer-diseases-3-class"
DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_ID))
print("Dataset root:", DATASET_ROOT)
print("Top-level contents:")
for p in sorted(DATASET_ROOT.iterdir()):
    print(" -", p)

In [ ]:

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def canonicalize_class(name: str):
    key = name.strip().lower().replace("_", " ").replace("-", " ")
    key = " ".join(key.split())
    if key in {"control", "healthy", "normal", "nc"}:
        return "CONTROL"
    if key in {"ad", "alzheimer", "alzheimer disease", "alzheimer's disease"}:
        return "AD"
    if key in {"pd", "parkinson", "parkinson disease", "parkinson's disease"}:
        return "PD"
    return None

class_dirs = {}
for p in DATASET_ROOT.rglob("*"):
    if p.is_dir():
        cls = canonicalize_class(p.name)
        if cls and cls not in class_dirs:
            class_dirs[cls] = p

missing = set(CLASS_NAMES) - set(class_dirs)
if missing:
    raise FileNotFoundError(f"Could not locate class directories for: {sorted(missing)}. Found: {class_dirs}")

print("Detected class directories:")
for cls in CLASS_NAMES:
    print(f"{cls}: {class_dirs[cls]}")

records = []
for cls in CLASS_NAMES:
    files = [p for p in class_dirs[cls].rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    print(cls, "images:", len(files))
    for p in files:
        records.append({"filepath": str(p), "label": cls})

df = pd.DataFrame(records)
print("Total images found:", len(df))
display(df.head())

In [ ]:

class_totals = df["label"].value_counts().reindex(CLASS_NAMES)
print(class_totals)

expected_totals = {
    "CONTROL": 2937 + 367 + 368,
    "AD": 2560 + 320 + 320,
    "PD": 774 + 97 + 96,
}
print("\nPaper Table I expected totals:", expected_totals)

if not all(class_totals[k] == expected_totals[k] for k in CLASS_NAMES):
    print("WARNING: The downloaded dataset counts do not exactly match the paper's Table I.\n"
          "The split cell below will stop rather than silently changing the class distribution.")

In [ ]:

def exact_stratified_split(frame: pd.DataFrame, target_counts: dict, seed: int = 42):
    rng = np.random.RandomState(seed)
    pieces = {"train": [], "validation": [], "test": []}

    for cls in CLASS_NAMES:
        part = frame[frame["label"] == cls].sample(frac=1.0, random_state=seed).reset_index(drop=True)
        targets = target_counts[cls]
        needed = targets["train"] + targets["validation"] + targets["test"]
        if len(part) != needed:
            raise ValueError(
                f"{cls}: found {len(part)} images but paper Table I requires {needed}. "
                "Please verify the downloaded dataset/version before continuing."
            )

        n_train = targets["train"]
        n_val = targets["validation"]
        pieces["train"].append(part.iloc[:n_train])
        pieces["validation"].append(part.iloc[n_train:n_train+n_val])
        pieces["test"].append(part.iloc[n_train+n_val:n_train+n_val+targets["test"]])

    out = {}
    for split in pieces:
        out[split] = pd.concat(pieces[split], ignore_index=True)
        out[split] = out[split].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

splits = exact_stratified_split(df, TARGET_COUNTS, SEED)

for split_name, split_df in splits.items():
    print(f"\n{split_name.upper()}: {len(split_df)}")
    print(split_df["label"].value_counts().reindex(CLASS_NAMES))

train_df = splits["train"]
val_df = splits["validation"]
test_df = splits["test"]

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, cls in zip(axes, CLASS_NAMES):
    row = train_df[train_df["label"] == cls].iloc[0]
    img = cv2.cvtColor(cv2.imread(row["filepath"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img, cmap="gray")
    ax.set_title(cls)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:

def gamma_clahe_image(img, clip=CLAHE_CLIP, grid=CLAHE_GRID, gamma=GAMMA):
    """Apply the paper's LAB -> CLAHE(L) -> Gamma(L) -> RGB pipeline.

    Input may be RGB uint8 (0..255) or float (also assumed to represent 0..255).
    Output is float32 RGB normalized to 0..1, matching the paper.
    """
    img = np.asarray(img)
    if img.ndim != 3:
        raise ValueError(f"Expected HxWx3 image, got {img.shape}")

    img8 = np.clip(img, 0, 255).astype(np.uint8)
    bgr = cv2.cvtColor(img8, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=tuple(grid))
    L_clahe = clahe.apply(L)


    L_gamma = np.clip(255.0 * np.power(L_clahe.astype(np.float32) / 255.0, gamma), 0, 255).astype(np.uint8)

    merged = cv2.merge([L_gamma, A, B])
    out_bgr = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
    return out_rgb.astype(np.float32) / 255.0


row = train_df.iloc[0]
raw = cv2.cvtColor(cv2.imread(row["filepath"]), cv2.COLOR_BGR2RGB)
processed = gamma_clahe_image(raw)
print("Raw:", raw.shape, raw.dtype, raw.min(), raw.max())
print("Processed:", processed.shape, processed.dtype, processed.min(), processed.max())

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for col, cls in enumerate(CLASS_NAMES):
    row = train_df[train_df["label"] == cls].iloc[0]
    raw = cv2.cvtColor(cv2.imread(row["filepath"]), cv2.COLOR_BGR2RGB)
    proc = gamma_clahe_image(raw)
    axes[0, col].imshow(raw, cmap="gray")
    axes[0, col].set_title(f"{cls} - Original")
    axes[1, col].imshow(proc)
    axes[1, col].set_title(f"{cls} - Gamma-CLAHE")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()

In [ ]:

train_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: gamma_clahe_image(x, CLAHE_CLIP, CLAHE_GRID, GAMMA),
    rotation_range=5,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05,
    fill_mode="nearest",
)

val_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: gamma_clahe_image(x, CLAHE_CLIP, CLAHE_GRID, GAMMA)
)

if PREPROCESS_TEST:
    test_datagen = ImageDataGenerator(
        preprocessing_function=lambda x: gamma_clahe_image(x, CLAHE_CLIP, CLAHE_GRID, GAMMA)
    )
else:
    test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="filepath", y_col="label",
    classes=CLASS_NAMES, class_mode="categorical",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)

val_gen = val_datagen.flow_from_dataframe(
    val_df, x_col="filepath", y_col="label",
    classes=CLASS_NAMES, class_mode="categorical",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

test_gen = test_datagen.flow_from_dataframe(
    test_df, x_col="filepath", y_col="label",
    classes=CLASS_NAMES, class_mode="categorical",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

print("Class indices:", train_gen.class_indices)
print("Train/Val/Test:", train_gen.samples, val_gen.samples, test_gen.samples)

In [ ]:

batch_x, batch_y = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.ravel(), batch_x[:8], batch_y[:8]):
    ax.imshow(image)
    ax.set_title(CLASS_NAMES[int(np.argmax(label))])
    ax.axis("off")
plt.tight_layout()
plt.show()

## InceptionV3 model


In [ ]:

def build_inceptionv3_model(input_shape=INPUT_SHAPE, num_classes=3, dropout=DROPOUT):
    base_model = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    base_model.trainable = False

    x = GlobalAveragePooling2D(name="global_average_pooling")(base_model.output)
    x = Dropout(dropout, name="dropout_0_5")(x)
    outputs = Dense(num_classes, activation="softmax", name="classifier")(x)

    model = Model(inputs=base_model.input, outputs=outputs, name="Gamma_CLAHE_InceptionV3")
    if LEARNING_RATE is None:
        optimizer = Adam()
    else:
        optimizer = Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base_model

model, inception_base = build_inceptionv3_model()
model.summary()

In [ ]:

CHECKPOINT_PATH = "/content/best_gamma_clahe_inceptionv3.keras"

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=6,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, reduce_lr, checkpoint]

In [ ]:

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:

def plot_history(history):
    hist = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist.get("accuracy", []), label="Train Acc")
    axes[0].plot(hist.get("val_accuracy", []), label="Val Acc")
    axes[0].set_title("Model Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(hist.get("loss", []), label="Train Loss")
    axes[1].plot(hist.get("val_loss", []), label="Val Loss")
    axes[1].set_title("Model Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history)

## Evaluation


In [ ]:

def evaluate_generator(model, generator, split_name):
    generator.reset()
    probs = model.predict(generator, verbose=1)
    y_true = generator.classes
    y_pred = np.argmax(probs, axis=1)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\n===== {split_name.upper()} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f} (macro)")
    print(f"Recall   : {recall:.4f} (macro)")
    print(f"F1-score : {f1:.4f} (macro)")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix - {split_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    return {
        "split": split_name,
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs,
        "cm": cm,
    }

val_results = evaluate_generator(model, val_gen, "Validation")
test_results = evaluate_generator(model, test_gen, "Test")

In [ ]:

def plot_multiclass_roc(result, title):
    y_true = result["y_true"]
    probs = result["probs"]
    y_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))

    plt.figure(figsize=(7, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], "k--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

plot_multiclass_roc(val_results, "ROC Curve - Validation")
plot_multiclass_roc(test_results, "ROC Curve - Test")

## Comparison with other pretrained models


In [ ]:

def build_transfer_baseline(backbone_fn, name, input_shape=INPUT_SHAPE, num_classes=3):
    base = backbone_fn(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = GlobalAveragePooling2D(name=f"{name}_gap")(base.output)
    x = Dropout(DROPOUT, name=f"{name}_dropout")(x)
    out = Dense(num_classes, activation="softmax", name=f"{name}_classifier")(x)
    m = Model(base.input, out, name=name)
    m.compile(optimizer=Adam(), loss="categorical_crossentropy", metrics=["accuracy"])
    return m

BACKBONES = {
    "EfficientNetB0": EfficientNetB0,
    "DenseNet121": DenseNet121,
    "ResNet50": ResNet50,
    "ResNet101": ResNet101,
}

print("Baselines prepared:", list(BACKBONES))

In [ ]:


baseline_rows = []
for name, backbone_fn in BACKBONES.items():
    print("\n" + "="*80)
    print("Training baseline:", name)
    baseline_model = build_transfer_baseline(backbone_fn, name)
    baseline_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=[
            EarlyStopping(monitor="val_accuracy", patience=10, mode="max", restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-7, verbose=0),
        ],
        verbose=1
    )

    res = evaluate_generator(baseline_model, test_gen, f"{name} Test")
    baseline_rows.append({
        "Model": name,
        "Accuracy": res["accuracy"],
        "Precision": res["precision_macro"],
        "Recall": res["recall_macro"],
        "F1": res["f1_macro"],
    })

baseline_rows.append({
    "Model": "Proposed InceptionV3",
    "Accuracy": test_results["accuracy"],
    "Precision": test_results["precision_macro"],
    "Recall": test_results["recall_macro"],
    "F1": test_results["f1_macro"],
})

baseline_df = pd.DataFrame(baseline_rows).sort_values("Model")
display(baseline_df)

## Hybrid preprocessing comparison


In [ ]:

def clahe_on_l_channel(img, clip=CLAHE_CLIP, grid=CLAHE_GRID, prefilter=None):
    img = np.asarray(img)
    img8 = np.clip(img, 0, 255).astype(np.uint8)
    if prefilter == "bilateral":
        img8 = cv2.bilateralFilter(img8, d=5, sigmaColor=50, sigmaSpace=50)
    elif prefilter == "median":
        img8 = cv2.medianBlur(img8, 5)

    bgr = cv2.cvtColor(img8, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=tuple(grid))
    L2 = clahe.apply(L)
    merged = cv2.merge([L2, A, B])
    out = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    out = cv2.cvtColor(out, cv2.COLOR_BGR2RGB)
    return out.astype(np.float32) / 255.0

print("Explicit alternatives ready: CLAHE, CLAHE+bilateral, CLAHE+median, Gamma-CLAHE")

In [ ]:
# Cell 24 - Factory for gamma-specific generators
def make_gamma_generators(gamma_value):
    train_dg = ImageDataGenerator(
        preprocessing_function=lambda x: gamma_clahe_image(x, CLAHE_CLIP, CLAHE_GRID, gamma_value),
        rotation_range=5, width_shift_range=0.05, height_shift_range=0.05,
        zoom_range=0.05, fill_mode="nearest"
    )
    val_dg = ImageDataGenerator(
        preprocessing_function=lambda x: gamma_clahe_image(x, CLAHE_CLIP, CLAHE_GRID, gamma_value)
    )
    tr = train_dg.flow_from_dataframe(
        train_df, x_col="filepath", y_col="label", classes=CLASS_NAMES,
        class_mode="categorical", target_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
    )
    va = val_dg.flow_from_dataframe(
        val_df, x_col="filepath", y_col="label", classes=CLASS_NAMES,
        class_mode="categorical", target_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
    )
    return tr, va



## Grad-CAM


In [ ]:

def find_last_conv_layer_name(base_model):
    for layer in reversed(base_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found for Grad-CAM")

LAST_CONV_LAYER = find_last_conv_layer_name(inception_base)
print("Grad-CAM layer:", LAST_CONV_LAYER)

In [ ]:

def make_gradcam_heatmap(img_batch, model, last_conv_layer_name, pred_index=None):
    base = model.get_layer("inceptionv3")
    conv_layer = base.get_layer(last_conv_layer_name)
    grad_model = tf.keras.models.Model(
        model.inputs,
        [conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
    return heatmap.numpy(), int(pred_index.numpy())


def overlay_gradcam(original_rgb, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * np.clip(heatmap, 0, 1))
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(original_rgb.astype(np.uint8), 1 - alpha, heatmap_color, alpha, 0)
    return overlay

In [ ]:

def load_for_model(path, preprocess_test=False):
    raw = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    resized = cv2.resize(raw, IMG_SIZE, interpolation=cv2.INTER_AREA)
    if preprocess_test:
        arr = gamma_clahe_image(resized)
    else:
        arr = resized.astype(np.float32) / 255.0
    return raw, arr

examples = []
for cls in CLASS_NAMES:
    rows = test_df[test_df["label"] == cls].head(2)
    for _, row in rows.iterrows():
        examples.append((cls, row["filepath"]))

fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4 * len(examples)))
if len(examples) == 1:
    axes = np.expand_dims(axes, 0)

for r, (true_cls, path) in enumerate(examples):
    raw, arr = load_for_model(path, PREPROCESS_TEST)
    resized_raw = cv2.resize(raw, IMG_SIZE, interpolation=cv2.INTER_AREA)
    batch = np.expand_dims(arr, axis=0)

    probs = model.predict(batch, verbose=0)[0]
    pred_id = int(np.argmax(probs))
    heatmap, _ = make_gradcam_heatmap(batch, model, LAST_CONV_LAYER, pred_id)
    overlay = overlay_gradcam(resized_raw, heatmap)

    axes[r, 0].imshow(resized_raw, cmap="gray")
    axes[r, 0].set_title(f"True: {true_cls}")
    axes[r, 1].imshow(heatmap, cmap="jet")
    axes[r, 1].set_title(f"Grad-CAM: {CLASS_NAMES[pred_id]}")
    axes[r, 2].imshow(overlay)
    axes[r, 2].set_title(f"Grad-CAM overlay ({probs[pred_id]:.3f})")
    for c in range(3):
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:

FINAL_MODEL_PATH = "/content/Gamma_CLAHE_InceptionV3_final.keras"
model.save(FINAL_MODEL_PATH)

summary = {
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "clahe_clip": CLAHE_CLIP,
    "clahe_grid": CLAHE_GRID,
    "gamma": GAMMA,
    "dropout": DROPOUT,
    "preprocess_test": PREPROCESS_TEST,
    "validation": {k: v for k, v in val_results.items() if k in {"accuracy", "precision_macro", "recall_macro", "f1_macro"}},
    "test": {k: v for k, v in test_results.items() if k in {"accuracy", "precision_macro", "recall_macro", "f1_macro"}},
}
print(json.dumps(summary, indent=2))
print("Saved:", FINAL_MODEL_PATH)